# Experimento 2: Evolución Temporal de Tópicos

**Pregunta:** ¿Cómo varía la distribución de tópicos entre períodos parlamentarios? ¿Se pueden identificar aparición, crecimiento y desaparición de temas en función del contexto político y social?

**Input:** Modelo BERTopic guardado en `data/bertopic_diputados_final` + `data/intervenciones_limpias.parquet`  
**Output:** Visualizaciones de prevalencia de tópicos × año y × período presidencial

## 1. Imports y carga del modelo

In [23]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Cargando modelo BERTopic...")
topic_model = BERTopic.load("../data/bertopic_diputados_final", embedding_model=embedding_model)
print(f"Modelo cargado: {len(topic_model.get_topic_info()) - 1} tópicos")

Cargando modelo BERTopic...
Modelo cargado: 190 tópicos


## 2. Cargar datos

In [24]:
print("Cargando intervenciones limpias...")
df = pd.read_parquet("../data/intervenciones_limpias.parquet")
df['fecha'] = pd.to_datetime(df['fecha'])
df['anio'] = df['fecha'].dt.year
df = df.dropna(subset=['texto_limpio'])
df = df[df['texto_limpio'].str.strip() != '']

textos = df['texto_limpio'].tolist()

print(f"Documentos cargados: {len(textos)}")
print(f"Rango temporal: {df['fecha'].min().date()} → {df['fecha'].max().date()}")

Cargando intervenciones limpias...
Documentos cargados: 172417
Rango temporal: 1983-12-16 → 2026-04-08


## 3. Asignar tópicos (con caché)

In [25]:
EMBEDDINGS_PATH = Path("../data/embeddings_intervenciones.npy")
TOPICOS_PATH = Path("../data/intervenciones_con_topico.parquet")

if TOPICOS_PATH.exists():
    print("Cargando tópicos desde disco...")
    df_topicos = pd.read_parquet(
        TOPICOS_PATH,
        columns=['id_periodo', 'id_reunion', 'n_intervencion', 'topic']
    )
    # El parquet tiene una fila por chunk (no por intervención), deduplicar
    df_topicos = df_topicos.drop_duplicates(
        subset=['id_periodo', 'id_reunion', 'n_intervencion']
    )
    topico_map = df_topicos.set_index(
        ['id_periodo', 'id_reunion', 'n_intervencion']
    )['topic']
    df['topic'] = (
        df.set_index(['id_periodo', 'id_reunion', 'n_intervencion'])
        .index.map(topico_map)
        .values
    )
    del df_topicos, topico_map
    print(f"Tópicos cargados. Top 20:")
    print(df['topic'].value_counts().head(20))
else:
    if EMBEDDINGS_PATH.exists():
        print("Cargando embeddings desde disco...")
        embeddings = np.load(EMBEDDINGS_PATH)
    else:
        print("Calculando embeddings (tarda varios minutos)...")
        embeddings = embedding_model.encode(textos, show_progress_bar=True, batch_size=64)
        np.save(EMBEDDINGS_PATH, embeddings)
        print(f"Embeddings guardados: {embeddings.shape}")

    print("Asignando tópicos (transform)...")
    topics, _ = topic_model.transform(textos, embeddings=embeddings)
    df['topic'] = topics
    print(f"Listo. Top 20:")
    print(df['topic'].value_counts().head(20))

Cargando tópicos desde disco...
Tópicos cargados. Top 20:
topic
0      16294
27      6324
40      5166
23      4986
1       4880
109     3817
3       3152
65      3132
180     2692
16      2484
6       2353
8       2312
57      2277
2       2254
24      2037
11      2019
19      1985
91      1984
31      1887
96      1801
Name: count, dtype: int64


## 4. Selección de tópicos para el análisis

De los 190 tópicos del modelo, seleccionamos manualmente 26 temáticamente coherentes.
Se excluyen: ruido, artefactos OCR, nombres propios y tópicos procedimentales (votar, jurar, quórum, etc.).

In [26]:
TOPICOS_SELECCIONADOS = {
    1:   "Presupuesto / Finanzas públicas",
    6:   "Derecho penal",
    8:   "Impuestos / Fiscal",
    11:  "Trabajo / Laboral",
    12:  "Salud / Discapacidad",
    18:  "Jubilaciones / Previsional",
    5:   "Energía / Gas / Combustibles",
    129: "Derechos humanos / Terrorismo",
    30:  "Agropecuario / Ganadería",
    22:  "Defensa / Fuerzas militares",
}

topicos_ids = list(TOPICOS_SELECCIONADOS.keys())

# Renombrar tópicos en el modelo para que los gráficos de BERTopic
# muestren nuestras etiquetas legibles en vez de "presupuesto_gasto_ciento_millon"
topic_model.set_topic_labels(TOPICOS_SELECCIONADOS)

df_validos = df[df['topic'].isin(topicos_ids)].copy()
df_validos['topic_label'] = df_validos['topic'].map(TOPICOS_SELECCIONADOS)

# Años sin datos: 1991-2000 (períodos no digitalizados por la HCDN)
GAP_INICIO = 1991
GAP_FIN    = 2000

print(f"{len(topicos_ids)} tópicos seleccionados, {len(df_validos)} intervenciones.")
print(f"Nota: no hay datos entre {GAP_INICIO} y {GAP_FIN}.")

10 tópicos seleccionados, 18870 intervenciones.
Nota: no hay datos entre 1991 y 2000.


## 5. Topics over Time (BERTopic nativo)

In [ ]:
TOT_PATH = Path("../data/topics_over_time.csv")
timestamps_anio = df['fecha'].dt.to_period('Y').dt.to_timestamp().tolist()

if TOT_PATH.exists():
    print("Cargando topics_over_time desde disco...")
    topics_over_time = pd.read_csv(TOT_PATH, parse_dates=['Timestamp'])
else:
    print("Calculando Topics over Time...")
    topics_over_time = topic_model.topics_over_time(
        textos,
        timestamps_anio,
        datetime_format="%Y-%m-%d",
        evolution_tuning=True,
        global_tuning=False
    )
    topics_over_time.to_csv(TOT_PATH, index=False)

top5_ids = df_validos['topic'].value_counts().head(5).index.tolist()

fig = topic_model.visualize_topics_over_time(
    topics_over_time,
    topics=top5_ids,
    custom_labels=True,
    title="Evolución de tópicos en el tiempo — Cámara de Diputados"
)

fig.add_vrect(
    x0="1991-01-01", x1="2001-01-01",
    fillcolor="lightgray", opacity=0.4, line_width=0,
    annotation_text="Sin datos<br>(1991-2000)",
    annotation_position="top left",
    annotation_font_size=10,
)

for trace in fig.data:
    if hasattr(trace, 'hovertemplate'):
        trace.hovertemplate = (
            "<b>%{fullData.name}</b><br>"
            "Año: %{x|%Y}<br>"
            "Frecuencia: %{y:.4f}<extra></extra>"
        )

fig.show()
fig.write_html("figuras/exp2/topics_over_time_bertopic.html")

## 6. Heatmap: prevalencia de tópicos por año

In [ ]:
pivot = (
    df_validos
    .groupby(['anio', 'topic_label'])
    .size()
    .reset_index(name='count')
)
pivot['prop'] = pivot.groupby('anio')['count'].transform(lambda x: x / x.sum())

heatmap_df = pivot.pivot(index='topic_label', columns='anio', values='prop').fillna(0)

fig_heat = px.imshow(
    heatmap_df,
    labels=dict(x="Año", y="Tópico", color="Proporción"),
    title="Prevalencia de tópicos por año — Cámara de Diputados",
    color_continuous_scale="Blues",
    aspect="auto",
    height=800
)
fig_heat.show()
fig_heat.write_html("figuras/exp2/heatmap_topicos_por_anio.html")

## 7. Heatmap: prevalencia por período presidencial

In [ ]:
def asignar_periodo(fecha):
    anio = fecha.year
    if anio <= 1989:
        return "Alfonsín (1983-1989)"
    elif anio <= 2001:
        return "Menem / De la Rúa (1989-2001)"
    elif anio <= 2003:
        return "Crisis / Transición (2001-2003)"
    elif anio <= 2007:
        return "N. Kirchner (2003-2007)"
    elif anio <= 2015:
        return "CFK (2007-2015)"
    elif anio <= 2019:
        return "Macri (2015-2019)"
    elif anio <= 2023:
        return "Fernández (2019-2023)"
    else:
        return "Milei (2023-2026)"

ORDEN_PERIODOS = [
    "Alfonsín (1983-1989)",
    "Menem / De la Rúa (1989-2001)",
    "Crisis / Transición (2001-2003)",
    "N. Kirchner (2003-2007)",
    "CFK (2007-2015)",
    "Macri (2015-2019)",
    "Fernández (2019-2023)",
    "Milei (2023-2026)",
]

df_validos['periodo_presidencial'] = df_validos['fecha'].apply(asignar_periodo)

pivot_pres = (
    df_validos
    .groupby(['periodo_presidencial', 'topic_label'])
    .size()
    .reset_index(name='count')
)
pivot_pres['prop'] = pivot_pres.groupby('periodo_presidencial')['count'].transform(lambda x: x / x.sum())

heatmap_pres = pivot_pres.pivot(
    index='topic_label', columns='periodo_presidencial', values='prop'
).fillna(0)

cols_ordenadas = [c for c in ORDEN_PERIODOS if c in heatmap_pres.columns]
heatmap_pres = heatmap_pres[cols_ordenadas]

fig_pres = px.imshow(
    heatmap_pres,
    labels=dict(x="Período presidencial", y="Tópico", color="Proporción"),
    title="Prevalencia de tópicos por período presidencial",
    color_continuous_scale="Blues",
    aspect="auto",
    height=800
)
fig_pres.update_xaxes(tickangle=90)
fig_pres.show()
fig_pres.write_html("figuras/exp2/heatmap_topicos_por_periodo_presidencial.html")

## 8. Heatmap: prevalencia por evento histórico

Para cada evento tomamos el año del evento + el año siguiente como ventana de análisis,
de modo de capturar el debate parlamentario que generó ese hito.

In [ ]:
EVENTOS_HISTORICOS = {
    1985: "Juicio Juntas",
    1989: "Hiperinflación",
    2001: "Crisis",
    2008: "Res. 125",
    2010: "Mat. igualitario",
    2012: "YPF",
    2018: "IVE",
    2020: "COVID-19",
    2024: "Ley Bases",
}

filas_evento = []
for anio_evento, nombre_evento in EVENTOS_HISTORICOS.items():
    ventana = df_validos[df_validos['anio'].isin([anio_evento, anio_evento + 1])]
    if len(ventana) == 0:
        continue
    conteos = ventana['topic_label'].value_counts()
    props = conteos / conteos.sum()
    for topico, prop in props.items():
        filas_evento.append({
            'evento': f"{nombre_evento}\n({anio_evento}–{anio_evento+1})",
            'topic_label': topico,
            'prop': prop
        })

df_eventos = pd.DataFrame(filas_evento)

orden_eventos = [
    f"{nombre}\n({anio}–{anio+1})"
    for anio, nombre in sorted(EVENTOS_HISTORICOS.items())
]
heatmap_eventos = (
    df_eventos
    .pivot(index='topic_label', columns='evento', values='prop')
    .fillna(0)
    .reindex(columns=[c for c in orden_eventos if c in df_eventos['evento'].unique()])
)

fig_ev = px.imshow(
    heatmap_eventos,
    labels=dict(x="Evento histórico", y="Tópico", color="Proporción"),
    title="Prevalencia de tópicos por evento histórico (ventana: año del evento + siguiente)",
    color_continuous_scale="Reds",
    aspect="auto",
    height=700,
)
fig_ev.update_xaxes(tickangle=90)
fig_ev.show()
fig_ev.write_html("figuras/exp2/heatmap_topicos_por_evento_historico.html")

## 10. Guardar resultados

In [32]:
df[['id_periodo', 'id_reunion', 'fecha', 'anio', 'orador', 'n_intervencion', 'topic']].to_parquet(
    "../data/intervenciones_con_topico.parquet", index=False
)
print("Guardado: data/intervenciones_con_topico.parquet")

Guardado: data/intervenciones_con_topico.parquet
